# Lab01 - Mật mã học cổ điển
**Nhóm 16 - NT101.R11.2**  
**Hà Nguyễn Hiếu An (24520043), Nguyễn Hữu Nghĩa (24521144)**

# Task 2.3

## Tự động tìm khóa
1. Khởi tạo theo thứ tự tần suất chữ; các lần tiếp theo làm nhiễu hoặc xáo trộn khóa.
2. Chấm điểm: tổng log10 xác suất quadgram + 2 × tổng log10 xác suất từ (từ dài ít nhất 2 chữ). Chuẩn hóa điểm bằng số quadgram. **Điểm cao hơn tốt hơn**, không phải xác suất đúng.
3. Đổi chỗ hai chữ trong khóa. Simulated annealing nhận bước tốt; có thể nhận bước kém với xác suất `exp(delta / T)`. Nhiệt độ giảm dần.
4. Khởi động lại nhiều lần, giữ khóa tốt nhất; hill-climbing tinh chỉnh bằng 325 phép đổi chỗ.
5. In `C =`, `p =`, khóa ánh xạ, IC, điểm, số ứng viên, seed và thời gian.

Quadgram chưa gặp dùng xác suất `0.01 / tổng số cụm`; từ chưa gặp dùng log10 xác suất `-12`. Trọng số từ 2.0 là lựa chọn heuristic, không được huấn luyện trên bản rõ Task 2.2.

**IC** = Σ nᵢ(nᵢ−1) / [N(N−1)]. Phép thay thế bảo toàn IC: dùng để quan sát, không dùng chọn khóa.

## Mô hình và giới hạn
Quadgram được thống kê từ 3 tác phẩm Project Gutenberg, sau khi bỏ header/footer; chữ được nối qua ranh giới từ. Tần suất từ lấy từ Peter Norvig. Dữ liệu nằm trong `task2_3_data`, chạy offline; mã chỉ dùng thư viện chuẩn Python.

Không bảo đảm tìm đúng khóa toàn cục. Văn bản ngắn, tên riêng, văn bản không phải tiếng Anh, hoặc mật mã không dùng một hoán vị cố định có thể cho kết quả sai. Không xác định chắc chắn ánh xạ chữ không xuất hiện. Để tìm thêm: tăng `restarts`, `iterations` hoặc đổi `seed`.

In [ ]:
import math
import random
import re
import time
from array import array
from collections import Counter
from pathlib import Path

MONO_ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MONO_FREQUENCY_ORDER = "ETAOINSHRDLCUMWFGYPBVKJXQZ"


def mono_data_path(filename):
    for folder in (Path.cwd(), Path.cwd() / "Lab1"):
        path = folder / "task2_3_data" / filename
        if path.is_file():
            return path
    raise FileNotFoundError("Không tìm thấy task2_3_data/" + filename + ". Chạy từ Lab1 hoặc thư mục gốc repository.")


def mono_normalize(text):
    return "".join(character.upper() for character in text
                   if "A" <= character <= "Z" or "a" <= character <= "z")


def mono_index_of_coincidence(text):
    counts = Counter(mono_normalize(text))
    total = sum(counts.values())
    return sum(n * (n - 1) for n in counts.values()) / (total * (total - 1)) if total > 1 else 0.0


def mono_validate_key(key):
    if not isinstance(key, str) or len(key) != 26 or set(key) != set(MONO_ALPHABET):
        raise ValueError("Khóa phải là hoán vị đủ 26 chữ hoa A-Z.")
    return key


def mono_decrypt(ciphertext, key):
    mono_validate_key(key)
    table = str.maketrans(MONO_ALPHABET + MONO_ALPHABET.lower(), key + key.lower())
    return ciphertext.translate(table)


def mono_encrypt(plaintext, key):
    mono_validate_key(key)
    inverse = "".join(MONO_ALPHABET[key.index(letter)] for letter in MONO_ALPHABET)
    return mono_decrypt(plaintext, inverse)


def mono_load_quadgrams(path):
    counts = {}
    for line in Path(path).read_text(encoding="ascii").splitlines():
        gram, count = line.split()
        if len(gram) != 4 or any(c not in MONO_ALPHABET for c in gram) or int(count) <= 0:
            raise ValueError("Dữ liệu quadgram không hợp lệ.")
        counts[gram] = int(count)
    if not counts:
        raise ValueError("Mô hình quadgram rỗng.")
    total = sum(counts.values())
    scores = array("d", [math.log10(0.01 / total)]) * (26 ** 4)
    for gram, count in counts.items():
        a, b, c, d = (ord(letter) - 65 for letter in gram)
        scores[((a * 26 + b) * 26 + c) * 26 + d] = math.log10(count / total)
    return {"scores": scores, "types": len(counts), "total": total}


def mono_load_words(path):
    counts = {}
    for line in Path(path).read_text(encoding="ascii").splitlines():
        word, count = line.split()
        counts[word] = int(count)
    total = sum(counts.values())
    if not total:
        raise ValueError("Mô hình từ rỗng.")
    return {word: math.log10(count / total) for word, count in counts.items()}


def mono_english_score(text, model):
    values = [ord(c) - 65 for c in mono_normalize(text)]
    if len(values) < 4:
        raise ValueError("Cần ít nhất 4 chữ cái ASCII để chấm điểm quadgram.")
    scores = model["scores"]
    score = sum(scores[((a * 26 + b) * 26 + c) * 26 + d]
                for a, b, c, d in zip(values, values[1:], values[2:], values[3:]))
    words = model.get("words", {})
    if words:
        score += sum(2.0 * words.get(word, -12.0)
                     for word in re.findall(r"[A-Za-z]+", text.upper()) if len(word) >= 2)
    return score / (len(values) - 3)


def mono_break(ciphertext, model, restarts=8, iterations=6000, seed=16):
    for value in (restarts, iterations):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError("Số lần khởi động và số vòng phải là số nguyên dương.")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise ValueError("Seed phải là số nguyên.")
    normalized = mono_normalize(ciphertext)
    if len(normalized) < 4:
        raise ValueError("Ciphertext cần ít nhất 4 chữ cái ASCII.")
    started = time.perf_counter()
    rng = random.Random(seed)
    letters = [ord(c) - 65 for c in normalized]
    frequencies = Counter(letters)
    order = sorted(range(26), key=lambda x: (-frequencies[x], x))
    initial = [0] * 26
    for cipher_index, plain_letter in zip(order, MONO_FREQUENCY_ORDER):
        initial[cipher_index] = ord(plain_letter) - 65
    gram_counts = Counter(zip(letters, letters[1:], letters[2:], letters[3:]))
    grams = list(gram_counts)
    weights = [gram_counts[g] for g in grams]
    quadgram_count = len(grams)
    words = model.get("words", {})
    if words:
        word_counts = Counter(tuple(ord(c) - 65 for c in word)
                              for word in re.findall(r"[A-Z]+", ciphertext.upper()) if len(word) >= 2)
        grams.extend(word_counts)
        weights.extend(word_counts.values())
    affected = [set() for _ in range(26)]
    for index, gram in enumerate(grams):
        for letter in set(gram):
            affected[letter].add(index)
    pairs = [(a, b, tuple(sorted(affected[a] | affected[b])))
             for a in range(26) for b in range(a + 1, 26)]
    scores = model["scores"]
    def gram_score(index, key):
        if index >= quadgram_count:
            word = "".join(MONO_ALPHABET[key[c]] for c in grams[index])
            return 2.0 * words.get(word, -12.0) * weights[index]
        a, b, c, d = grams[index]
        return scores[((key[a] * 26 + key[b]) * 26 + key[c]) * 26 + key[d]] * weights[index]
    best_score = -math.inf
    best_key = None
    evaluated = 0
    history = []
    accepted = 0
    for restart in range(restarts):
        key = initial.copy()
        if restart % 3 == 2:
            rng.shuffle(key)
        elif restart:
            for _ in range(4 + restart):
                a, b = rng.sample(range(26), 2)
                key[a], key[b] = key[b], key[a]
        current = [gram_score(i, key) for i in range(len(grams))]
        score = sum(current)
        if score > best_score:
            best_score, best_key = score, key.copy()
        temperature = 0.015 * (len(letters) - 3)
        cooling = (0.0002 / 0.015) ** (1 / max(iterations - 1, 1))
        for _ in range(iterations):
            a, b, indices = rng.choice(pairs)
            key[a], key[b] = key[b], key[a]
            proposed = [gram_score(i, key) for i in indices]
            delta = sum(new - current[i] for i, new in zip(indices, proposed))
            evaluated += 1
            if delta >= 0 or rng.random() < math.exp(delta / temperature):
                for i, value in zip(indices, proposed):
                    current[i] = value
                score += delta
                accepted += 1
                if score > best_score:
                    best_score, best_key = score, key.copy()
            else:
                key[a], key[b] = key[b], key[a]
            temperature *= cooling
        history.append(best_score / (len(letters) - 3))
    key = best_key.copy()
    current = [gram_score(i, key) for i in range(len(grams))]
    while True:
        best_delta, move = 1e-9, None
        for a, b, indices in pairs:
            key[a], key[b] = key[b], key[a]
            proposed = [gram_score(i, key) for i in indices]
            delta = sum(new - current[i] for i, new in zip(indices, proposed))
            evaluated += 1
            key[a], key[b] = key[b], key[a]
            if delta > best_delta:
                best_delta, move = delta, (a, b, indices, proposed)
        if move is None:
            break
        a, b, indices, proposed = move
        key[a], key[b] = key[b], key[a]
        for i, value in zip(indices, proposed):
            current[i] = value
    key_text = "".join(MONO_ALPHABET[i] for i in key)
    plaintext = mono_decrypt(ciphertext, key_text)
    return {"plaintext": plaintext, "key": key_text,
            "score": mono_english_score(plaintext, model), "letters": len(letters),
            "restarts": restarts, "iterations": iterations, "evaluated": evaluated,
            "accepted": accepted, "seed": seed, "seconds": time.perf_counter() - started,
            "history": history}


def mono_read_ciphertext(path):
    return Path(path).expanduser().read_text(encoding="utf-8-sig")


def mono_show_result(ciphertext, result):
    print("C =", ciphertext)
    print("p =", result["plaintext"])
    print("Ánh xạ ciphertext -> plaintext:")
    print("C:", MONO_ALPHABET)
    print("p:", result["key"])
    print("Số chữ ASCII:", result["letters"])
    print("Index of Coincidence:", round(mono_index_of_coincidence(ciphertext), 6))
    print("Điểm ngôn ngữ chuẩn hóa (cao hơn tốt hơn):", round(result["score"], 6))
    print("Số lần khởi động:", result["restarts"])
    print("Số vòng mỗi lần:", result["iterations"])
    print("Số khóa ứng viên đã đánh giá:", result["evaluated"])
    print("Số bước annealing chấp nhận:", result["accepted"])
    print("Seed:", result["seed"])
    print("Thời gian (giây):", round(result["seconds"], 3))
    if result["letters"] < 100:
        print("Cảnh báo: văn bản ngắn, thống kê ít; bản rõ có thể sai.")


### thống kê

In [2]:
MONO_MODEL = mono_load_quadgrams(mono_data_path("english_quadgrams.txt"))
MONO_MODEL["words"] = mono_load_words(mono_data_path("english_words.txt"))
print("Số loại quadgram:", MONO_MODEL["types"])
print("Tổng số quadgram:", MONO_MODEL["total"])
print("Số từ trong mô hình:", len(MONO_MODEL["words"]))

Số loại quadgram: 48513
Tổng số quadgram: 1327769
Số từ trong mô hình: 333333


### Giải ciphertext Task 2.2

In [3]:
MONO_TASK_CIPHERTEXT = mono_read_ciphertext(mono_data_path("cipher2_2.txt"))
MONO_TASK_RESULT = mono_break(MONO_TASK_CIPHERTEXT, MONO_MODEL,
                              restarts=4, iterations=4000, seed=16)
mono_show_result(MONO_TASK_CIPHERTEXT, MONO_TASK_RESULT)
print("Mã hóa lại khớp ciphertext:",
      mono_encrypt(MONO_TASK_RESULT["plaintext"], MONO_TASK_RESULT["key"]) == MONO_TASK_CIPHERTEXT)
print("Lưu ý: mã hóa lại khớp chỉ kiểm tra tính nhất quán, không chứng minh bản rõ đúng.")

C = Ghv Hliib Kzggvi svirvs, uirggvm yb Yirgrsh lfghzi Q.P. Izuormt, rs zmv zd ghv nzsg kzkfoli lmw rmdofvmgrlo dlmglsb sltls zd nzwvim orgvilgfiv. Sklmmrmt svevm yzzps, ghv svirvs dzoozus ghv ordv zd l bzfmt yzb, Hliib Kzggvi, uhz wrsxzevis zm hrs 11gh yrighwlb ghlg hv rs l urcliw. Hv rs rmergvw gz lggvmw Hztuligs Sxhzzo zd Urgxhxildg lmw Urcliwib, uhviv hv ovlims lyzfg hrs nltrxlo hvirgltv, wvevozks xozsv dirvmwshrks, lmw xzmdizmgs ghv wlip dzixvs ghlg ghivlgvm ghv urcliwrmt uziow.



Ghv xvmgilo kozg ivezoevs lizfmw Hliib's zmtzrmt ylggov urgh Oziw Ezowvnzig, l wlip urcliw rmgvmg zm xzmjfvirmt yzgh ghv nltrxlo lmw mzm-nltrxlo uziows. Ls Hliib tizus zowvi, hv ovlims nziv lyzfg hrs zum nbsgvirzfs klsg, hrs xzmmvxgrzm gz Ezowvnzig, lmw ghv kizkhvxb ghlg grvs ghvri dlgvs gztvghvi. Ghizfthzfg ghv svirvs, ghvnvs zd dirvmwshrk, ozblogb, yilevib, lmw ghv kzuvi zd ozev liv vakozivw, urgh l irxh uziow droovw urgh nltrxlo xivlgfivs, skvoos, lmw wvvk nbghzoztb.



Ghv yzzps, sgligrmt urgh Hliib

### Nhập dữ liệu

In [4]:
MONO_INTERACTIVE = False

if MONO_INTERACTIVE:
    try:
        mono_mode = input("Monoalphabetic: 1 = nhập ciphertext, 2 = đọc file: ").strip()
        if mono_mode not in {"1", "2"}:
            raise ValueError("Chế độ phải là 1 hoặc 2.")
        print("Chế độ =", mono_mode)
        if mono_mode == "1":
            mono_input_text = input("C = ")
        else:
            mono_input_path = input("Đường dẫn file UTF-8 = ").strip()
            print("File =", mono_input_path)
            mono_input_text = mono_read_ciphertext(mono_input_path)
        print("C đã nhập =", mono_input_text)
        mono_restarts = int(input("Số lần khởi động [4] = ").strip() or "4")
        mono_iterations = int(input("Số vòng mỗi lần [4000] = ").strip() or "4000")
        mono_seed = int(input("Seed [16] = ").strip() or "16")
        mono_input_result = mono_break(mono_input_text, MONO_MODEL,
                                      mono_restarts, mono_iterations, mono_seed)
        mono_show_result(mono_input_text, mono_input_result)
    except (ValueError, OSError, UnicodeError) as error:
        print("Lỗi:", error)
else:
    print("Đổi MONO_INTERACTIVE = True để sử dụng.")

Đổi MONO_INTERACTIVE = True để sử dụng.


## Đánh giá phá mã trên văn bản độc lập

In [5]:
MONO_SAMPLE = 'The university library opens early every morning and welcomes students from different departments. Some readers prepare for examinations, while others search for information about science, history, and technology. A librarian explains how to evaluate reliable sources and compare evidence before drawing a conclusion. Outside the building, several friends discuss a research project about clean water and public health. They plan to collect observations, organize the results, and present their findings to the community. Good communication helps the team identify mistakes and improve its work. Although each student brings a different perspective, they share the same goal of learning useful skills and solving practical problems. Education becomes more meaningful when knowledge is connected with everyday experience.'
mono_sample_rng = random.Random(101)
MONO_SAMPLE_KEY = "".join(mono_sample_rng.sample(MONO_ALPHABET, 26))
MONO_BENCHMARKS = []
for label, original, restart_count, round_count, search_seed in (
    ("Dài, seed 16", MONO_SAMPLE, 4, 4000, 16),
    ("Dài, seed 29", MONO_SAMPLE, 4, 4000, 29),
    ("Ngắn, seed 16", "The library opens early every morning.", 4, 4000, 16),
):
    ciphertext = mono_encrypt(original, MONO_SAMPLE_KEY)
    found = mono_break(ciphertext, MONO_MODEL, restart_count, round_count, search_seed)
    expected_letters = mono_normalize(original)
    actual_letters = mono_normalize(found["plaintext"])
    correct = sum(a == b for a,b in zip(expected_letters, actual_letters))
    accuracy = correct / len(expected_letters)
    MONO_BENCHMARKS.append({"label": label, "correct": correct, "total": len(expected_letters),
                            "accuracy": accuracy, "result": found, "ciphertext": ciphertext})
    print("Test:", label)
    mono_show_result(ciphertext, found)
    print("Độ đúng chữ ASCII:", f"{correct}/{len(expected_letters)} = {accuracy:.4%}")
    print("Khớp toàn bộ bản rõ:", found["plaintext"] == original)
    print()

Test: Dài, seed 16
C = Tir qszxrcazth dzfcuch ekrsa rucdh rxrch mecszsb usn lrdwemra atqnrsta ocem nzoorcrst nrkuctmrsta. Aemr crunrca kcrkucr oec rpumzsutzesa, lizdr etirca arucwi oec zsoecmutzes ufeqt awzrswr, izatech, usn trwisedebh. U dzfcuczus rpkduzsa iel te rxudqutr crdzufdr aeqcwra usn wemkucr rxznrswr froecr nculzsb u weswdqazes. Eqtaznr tir fqzdnzsb, arxrcud oczrsna nzawqaa u crarucwi kcejrwt ufeqt wdrus lutrc usn kqfdzw irudti. Tirh kdus te weddrwt efarcxutzesa, ecbuszyr tir craqdta, usn kcrarst tirzc ozsnzsba te tir wemmqszth. Been wemmqszwutzes irdka tir trum znrstzoh mzatuvra usn zmkcexr zta lecv. Udtieqbi ruwi atqnrst fczsba u nzoorcrst krcakrwtzxr, tirh aiucr tir aumr beud eo drucszsb qaroqd avzdda usn aedxzsb kcuwtzwud kcefdrma. Rnqwutzes frwemra mecr mruszsboqd lirs vseldrnbr za wessrwtrn lzti rxrchnuh rpkrczrswr.
p = The university library opens early every morning and welcomes students from different departments. Some readers prepare for examinations, while others s

Test: Dài, seed 29
C = Tir qszxrcazth dzfcuch ekrsa rucdh rxrch mecszsb usn lrdwemra atqnrsta ocem nzoorcrst nrkuctmrsta. Aemr crunrca kcrkucr oec rpumzsutzesa, lizdr etirca arucwi oec zsoecmutzes ufeqt awzrswr, izatech, usn trwisedebh. U dzfcuczus rpkduzsa iel te rxudqutr crdzufdr aeqcwra usn wemkucr rxznrswr froecr nculzsb u weswdqazes. Eqtaznr tir fqzdnzsb, arxrcud oczrsna nzawqaa u crarucwi kcejrwt ufeqt wdrus lutrc usn kqfdzw irudti. Tirh kdus te weddrwt efarcxutzesa, ecbuszyr tir craqdta, usn kcrarst tirzc ozsnzsba te tir wemmqszth. Been wemmqszwutzes irdka tir trum znrstzoh mzatuvra usn zmkcexr zta lecv. Udtieqbi ruwi atqnrst fczsba u nzoorcrst krcakrwtzxr, tirh aiucr tir aumr beud eo drucszsb qaroqd avzdda usn aedxzsb kcuwtzwud kcefdrma. Rnqwutzes frwemra mecr mruszsboqd lirs vseldrnbr za wessrwtrn lzti rxrchnuh rpkrczrswr.
p = The university library opens early every morning and welcomes students from different departments. Some readers prepare for examinations, while others s

## Nhận xét Task 2.2 và giới hạn
Bản rõ tham chiếu trong báo cáo Task 2.2 và ciphertext tải từ đề có 4.578 chữ ASCII. Đối chiếu riêng phục vụ đánh giá cho thấy ánh xạ A/J sang Q/X không nhất quán ở 3 vị trí. Một khóa hoán vị cố định không thể khớp hoàn toàn bản rõ tham chiếu này. 

Kết quả seed 16, 4 lần × 4.000 vòng được so sánh trong báo cáo Word. Bản rõ tham chiếu không được truyền vào `mono_break`, không dùng tạo mô hình hoặc xếp hạng khóa.

Văn bản ngắn thường thiếu bằng chứng thống kê. Tăng số lần khởi động hoặc số vòng có thể cải thiện tìm kiếm nhưng không sửa được lỗi dữ liệu hoặc sai lệch mô hình. Văn bản không có khoảng trắng vẫn chấm quadgram; điểm từ khi ấy ít hữu ích. Kết quả tốt nhất chỉ là tốt nhất **trong các khóa đã khảo sát**.

## Tài liệu tham khảo
- Ciphertext Task 2.2: https://pastecode.io/s/h3hvp47p
- Quadgram từ Pride and Prejudice, The Adventures of Sherlock Holmes, Frankenstein: https://www.gutenberg.org/ebooks/1342 ; https://www.gutenberg.org/ebooks/1661 ; https://www.gutenberg.org/ebooks/84
- Điều kiện sử dụng Gutenberg: https://www.gutenberg.org/policy/license.html ; tác phẩm public domain tại Hoa Kỳ, cần kiểm tra luật địa phương.
- Tần suất từ: https://norvig.com/ngrams/ ; https://norvig.com/ngrams/count_1w.txt
- Simulated annealing: https://en.wikipedia.org/wiki/Simulated_annealing
- Nguồn và SHA-256 dữ liệu: `task2_3_data/sources.json`.